# 14 · Distilling DeBERTa-large into MiniLM (supplement N3 / T2)

**Question:** the final system needs 10 cross-encoders (5 DeBERTa-base and 5 DeBERTa-large fold models) per query. How much of the gap between a MiniLM cross-encoder (about 33M parameters) and DeBERTa-large does a MiniLM recover when it learns from the large model's class probabilities?

**Teacher:** the out-of-fold probabilities `p_E..p_I` of DeBERTa-large (5 folds, `12_deberta_large_t4x2`, `assemble`): every train pair is scored by the fold model that did not see its query, so the soft targets carry no label leakage.

**Student:** `cross-encoder/ms-marco-MiniLM-L-12-v2`, `main`, `max_len` 256, 4-class head, 1 epoch, batch 32, lr 7e-6, 5,000 warm-up steps (= `d1_minilm_main_mse` except the loss). Loss `ce4_soft`: cross-entropy on `alpha * one_hot(label) + (1 - alpha) * teacher`.

| Run | Loss | alpha | Seeds | Role |
|---|---|---|---|---|
| `d4_minilm_main_ce4` | ce4 (hard labels) | 1 | 42, 43 | reference; **checkpoint saved** (notebooks 16, 17) |
| `d4_minilm_main_kd05` | ce4_soft | 0.5 | 42, 43 | half label, half teacher |
| `d4_minilm_main_kd00` | ce4_soft | 0 | 42, 43 | teacher only |

The six runs are independent and run in parallel, one per GPU. **Best alpha** = the KD variant with the higher val nDCG at seed 42 (rule fixed in advance); its seed-43 run is the replication, the other one is reported too. Every full run saves its checkpoint (fp16, about 65 MB).

**Main number** (val now, test in `10_report`): recovered share = (student − ce4 reference) / (large 5-fold − ce4 reference), CI by bootstrap over queries. The comparison is with `d4_minilm_main_ce4` (same 4-class head), not with the MSE run `d1_minilm_main_mse` (one-output MS MARCO head).

**Kaggle settings:** GPU (every visible GPU is used: one run per GPU at a time, `src/gpu_jobs.py`). **Internet on**, or attach the output of `00_hf_models`.

**Inputs:**
- Dataset `wm-src` built from the **`supplement`** branch (loss `ce4_soft`, `src/gpu_jobs.py`). Required.
- Add Input → Notebook Output → **01_preprocess**. Required.
- Add Input → Notebook Output → **12_deberta_large_t4x2** (version run with `MODE = 'assemble'`): `oof/ce_debertalarge_main_ce4.parquet`. Required.
- Optional: Notebook Output → **00_hf_models** (session without Internet).

**How to run.**
1. `MODE = 'smoke'`: 5 % of train, seed 42, ce4 and kd00 in parallel, val only. Check section 2 and the verdict of section 3 (it prints the time of one full run).
2. `MODE = 'full'` → Save & Run All (6 runs, in rounds of one run per GPU).
3. Download `results/` into `data/results/supplement/` (not the models), and keep the notebook output as the checkpoint source for notebooks 16 and 17.

**Output** (`/kaggle/working/`): `results/<run>.json`, `results/scores/<run>_{val,test}.parquet` (with `p_E..p_I`), `results/d4_summary.csv`, `models/<run>/` (checkpoint), `jobs/<run>/log.txt`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, json, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import src.cross_encoder as ce
from src.cross_encoder import LABELS
from src.data import find, load_pairs
from src.evaluation import RunLog, calibration, paired_bootstrap, query_metrics, summarize

assert hasattr(ce, 'make_targets'), 'wm-src is not from the supplement branch: rebuild it'
from src.gpu_jobs import cleanup, prefetch, run_jobs, save_frame

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train, seed 42, val only | 'full'

BASE = {'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'main', 'max_len': 256,
        'epochs': 1, 'batch_size': 32, 'lr': 7e-6, 'warmup_steps': 5000,
        'weight_decay': 0.01, 'max_grad_norm': 1.0, 'fp16': True, 'eval_batch_size': 512}   # = d1_minilm_main
VARIANTS = {'ce4': {'loss': 'ce4'},
            'kd05': {'loss': 'ce4_soft', 'alpha': 0.5},
            'kd00': {'loss': 'ce4_soft', 'alpha': 0.0}}
SEEDS = [42] if MODE == 'smoke' else [42, 43]
JOBS = [('ce4', 42), ('kd00', 42)] if MODE == 'smoke' else [(v, s) for s in SEEDS for v in VARIANTS]   # ce4 first per seed

SEED = 42
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']     # test is predicted and saved, never scored here
EVAL_EVERY = 300 if MODE == 'smoke' else None
PREFIX = 'smoke_' if MODE == 'smoke' else ''
PROBS = [f'p_{l}' for l in LABELS]
SOFT = [f'soft_{l}' for l in LABELS]
TEACHER = 'ce_debertalarge_main_ce4.parquet'
RES, MODELS, WORK = Path('/kaggle/working/results'), Path('/kaggle/working/models'), Path('/kaggle/working/jobs')
DATA = Path('/tmp/wm_jobs_data')                                  # temporary data copies (not in the output)

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU'
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', N_GPU, 'x', torch.cuda.get_device_name(0))
print('MODE', MODE, '| jobs', JOBS)

## 1. Data and teacher

Pairs with `main` text; the teacher's out-of-fold probabilities are joined to the train pairs as `soft_E..soft_I`. Checks: every train pair has a teacher row, each row sums to 1, and the teacher's own val nDCG matches `d3_ce_debertalarge_main_ce4` (0.8970).

In [ ]:
pairs = load_pairs(BASE['fields'])
teacher = pd.read_parquet(find(TEACHER, "Add Input -> Notebook Output -> 12_deberta_large_t4x2 (MODE = 'assemble')"))
print('teacher', teacher.shape, teacher['split'].value_counts().to_dict())

train_df = pairs[pairs['split'] == 'train'].merge(
    teacher.loc[teacher['split'] == 'train', ['example_id'] + PROBS].rename(columns=dict(zip(PROBS, SOFT))),
    on='example_id', how='left', validate='1:1')
assert train_df[SOFT].notna().all().all(), 'train pairs without a teacher probability'
assert np.allclose(train_df[SOFT].sum(axis=1), 1, atol=1e-3), 'teacher rows do not sum to 1'
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[pairs['split'] == s].reset_index(drop=True) for s in ['val', 'test']}

t_val = parts['val'][['example_id']].merge(teacher[['example_id', 'score']], on='example_id', how='left')['score']
assert t_val.notna().all(), 'teacher has no val score for some pairs'
teacher_metrics, teacher_pq = summarize(parts['val'], t_val.to_numpy())
print(f"teacher (large, 5 fold models) val nDCG {teacher_metrics['ndcg']} (d3_ce_debertalarge_main_ce4: 0.8970)")
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs')

## 2. Loss self-check on the real model

On one batch of real pairs: `ce4_soft` with alpha = 1 gives the same loss as `ce4`, and with alpha = 0 the target rows are the teacher rows.

In [ ]:
src_model = prefetch(BASE['model'])   # downloaded once here; the parallel jobs load it from the cache
tok = AutoTokenizer.from_pretrained(src_model)
m = AutoModelForSequenceClassification.from_pretrained(src_model, num_labels=4, ignore_mismatched_sizes=True).to(ce.DEVICE)
b = train_df.iloc[:32]
with torch.no_grad():
    logits = m(**ce.encode(tok, b['query'], b['text'], BASE['max_len'])).logits.float()
y_hard = torch.from_numpy(ce.make_targets(b, {'loss': 'ce4'})).to(ce.DEVICE)
y_a1 = torch.from_numpy(ce.make_targets(b, {'loss': 'ce4_soft', 'alpha': 1.0})).to(ce.DEVICE)
assert torch.allclose(ce.compute_loss(logits, y_hard, {'loss': 'ce4'}),
                      ce.compute_loss(logits, y_a1, {'loss': 'ce4_soft'}), atol=1e-5)
assert np.allclose(ce.make_targets(b, {'loss': 'ce4_soft', 'alpha': 0.0}), b[SOFT].to_numpy(), atol=1e-5)
del m
torch.cuda.empty_cache()
print('ce4_soft self-check passed')

## 3. Training runs

Each run is one job of `src/gpu_jobs.py` (one GPU): train, predict val and test (with `p_E..p_I`), save the checkpoint in fp16 after predicting (full mode). The jobs run in parallel; when all are done, every run is logged here with ce4 first in each seed (paired bootstrap against the ce4 reference of the same seed).

In [ ]:
data = {'train': save_frame(train_df, DATA / 'train.parquet'),
        **{s: save_frame(parts[s], DATA / f'{s}.parquet') for s in EVAL_SPLITS}}
jobs = [{'name': f'{PREFIX}d4_minilm_main_{v}_s{seed}', 'kind': 'train', 'cfg': {**BASE, **VARIANTS[v]}, 'seed': seed,
         'train': data['train'], 'predict': {s: data[s] for s in EVAL_SPLITS}, 'eval_every': EVAL_EVERY,
         'print_every': 100 if MODE == 'smoke' else 1000,
         'save_dir': str(MODELS / f'{PREFIX}d4_minilm_main_{v}_s{seed}') if MODE == 'full' else None}
        for v, seed in JOBS]
results = run_jobs(jobs, WORK, N_GPU)

log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
runs, probs_val = [], {}
for job in jobs:
    name, cfg, seed, r = job['name'], job['cfg'], job['seed'], results[job['name']]
    ref = f'{PREFIX}d4_minilm_main_ce4_s{seed}' if cfg['loss'] != 'ce4' else None
    probs_val[name] = r['probs']['val']
    log.log(name, r['scores'],
            {'mode': MODE, **cfg, 'seed': seed, 'subsample': SUBSAMPLE, 'teacher': TEACHER,
             'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
             'n_gpus': 1, 'parallel_jobs': N_GPU, **r['info']},
            compare_to=ref, blind=('test',), extra=r['probs'])
    runs.append(name)
    if job['save_dir']:
        (Path(job['save_dir']) / 'checkpoint_info.json').write_text(json.dumps(
            {'run': name, 'cfg': cfg, 'seed': seed, 'val_ndcg': log.records[name]['metrics']['val']['ndcg'],
             'src_version': src_version()}, indent=1))
cleanup(WORK, DATA)

In [ ]:
if MODE == 'smoke':
    ok = True
    for n in runs:
        r = log.records[n]
        first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
        v = r['metrics']['val']['ndcg']
        good = last < first and v > 0.742
        ok &= good
        print(f"{'OK   ' if good else 'CHECK'} {n:34s} loss {first:.4f} -> {last:.4f} | val nDCG {v} | "
              f"full epoch ~{r['steps'] / SUBSAMPLE / r['it_per_s'] / 3600:.2f} h")
    print('Smoke test passed: switch MODE to full.' if ok else 'Some runs did not learn: send me this output.')

## 4. Results (val)

- Each run against the ce4 reference of the same seed.
- **Recovered share** of the gap to the teacher: (student − reference) / (teacher − reference), with a CI from the same bootstrap resample of queries for numerator and denominator.
- **Pooled over seeds 42, 43:** each KD variant − ce4, per query the mean over seeds, bootstrap over queries; and the best alpha by the seed-42 rule.
- Inference time per 1,000 pairs (`gpu_ms_per_1k_pairs`) is only indicative here: the runs share the machine's CPUs while they predict. Notebook 18 times every model alone on one GPU for the cost–quality plot.
- Calibration of every run (all have a 4-class head).

In [ ]:
def pooled_delta(pairs_of_runs, split='val', metric='ndcg', n_boot=1000, seed=42):
    """Paired difference b - a pooled over seeds: per query, the mean over seeds of (b_s - a_s); bootstrap over
    queries. pairs_of_runs: [(run_a, run_b), ...], one pair per seed, all in log.per_query.
    Returns the pooled diff and CI, the diff per seed and whether every seed has the same sign."""
    per_seed, frames = [], []
    for a, b in pairs_of_runs:
        p = paired_bootstrap(log.per_query[a][split], log.per_query[b][split], metric)
        per_seed.append(p['diff'])
        d = log.per_query[a][split][['query_id', metric]].merge(
            log.per_query[b][split][['query_id', metric]], on='query_id', suffixes=('_a', '_b')).dropna()
        frames.append(d.set_index('query_id').eval(f'{metric}_b - {metric}_a').rename(len(frames)))
    diff = pd.concat(frames, axis=1, join='inner').mean(axis=1).to_numpy()
    means = diff[np.random.default_rng(seed).integers(0, len(diff), (n_boot, len(diff)))].mean(axis=1)
    return {'diff': round(float(diff.mean()), 4),
            'ci95': [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)],
            'per_seed': per_seed, 'same_sign': len({np.sign(x) for x in per_seed}) == 1, 'n_queries': len(diff)}

In [ ]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
log.print_paired()


def recovered(student, reference, n_boot=1000, seed=42):
    s, r, t = (x[['query_id', 'ndcg']].set_index('query_id')['ndcg']
               for x in (log.per_query[student]['val'], log.per_query[reference]['val'], teacher_pq))
    d = pd.concat({'s': s, 'r': r, 't': t}, axis=1, join='inner').dropna().to_numpy()
    point = (d[:, 0].mean() - d[:, 1].mean()) / (d[:, 2].mean() - d[:, 1].mean())
    idx = np.random.default_rng(seed).integers(0, len(d), (n_boot, len(d)))
    m = d[idx].mean(axis=1)
    boot = (m[:, 0] - m[:, 1]) / (m[:, 2] - m[:, 1])
    return round(float(point), 3), [round(float(np.percentile(boot, 2.5)), 3), round(float(np.percentile(boot, 97.5)), 3)]


rows = []
for n in runs:
    r = log.records[n]
    seed = r['seed']
    ref = f'{PREFIX}d4_minilm_main_ce4_s{seed}'
    row = {'run': n, 'val_ndcg': r['metrics']['val']['ndcg'], 'e_over_s': r['metrics']['val']['e_over_s'],
           'gpu_ms_per_1k_pairs': round(r['predict_minutes'] * 60e3 / r['predict_pairs'] * 1000, 1),
           **calibration(probs_val[n].to_numpy(), parts['val']['esci_label'].to_numpy())}
    if n != ref and ref in log.per_query:
        row['vs_ce4'] = paired_bootstrap(log.per_query[ref]['val'], log.per_query[n]['val'])
        row['recovered'], row['recovered_ci95'] = recovered(n, ref)
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(RES / f'{PREFIX}d4_summary.csv', index=False)
print(f"teacher val nDCG {teacher_metrics['ndcg']}")

if MODE == 'full':
    pooled = pd.DataFrame([{'comparison': f'{v} - ce4 (seeds {SEEDS})',
                            **pooled_delta([(f'd4_minilm_main_ce4_s{s}', f'd4_minilm_main_{v}_s{s}') for s in SEEDS])}
                           for v in ('kd05', 'kd00')])
    display(pooled)
    pooled.to_csv(RES / 'd4_pooled.csv', index=False)
    s42 = {v: log.records[f'd4_minilm_main_{v}_s42']['metrics']['val']['ndcg'] for v in ('kd05', 'kd00')}
    best = max(s42, key=s42.get)
    print(f'best alpha by the seed-42 rule: {best} (val nDCG at seed 42: {s42}); '
          f'its replication is d4_minilm_main_{best}_s43')